In [ ]:

import os, sys, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
code_dir=os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
prev=os.path.dirname(glob.glob("/kaggle/input/**/features_human.npy", recursive=True)[0])
big=os.path.dirname(glob.glob("/kaggle/input/**/llm_pairs_2m.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
for p in glob.glob(code_dir+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.makedirs("/kaggle/working/models",exist_ok=True)
shutil.copy(code_dir+"/anti_words.json","/kaggle/working/models/anti_words.json")
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.hybrid import product_disjoint_pair_masks
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

pairs=pd.read_parquet(big+"/llm_pairs_2m.parquet")
items=pd.read_parquet(big+"/llm_items_2m.parquet")
log(f"2 млн пар: {len(pairs):,}, карточек {len(items):,}, доля+ {pairs['label'].mean():.3f}")

# Считаем по кускам категорий, чтобы не повторить падение по памяти на 1.5 млн карточек.
cat_of=dict(zip(items["id"].tolist(), items["category"].astype(str).tolist()))
pc=pairs["id1"].map(cat_of).fillna("?").astype(str).to_numpy()
names=feature_names(True)
X=np.zeros((len(pairs),len(names)),dtype=np.float32)
for cat in sorted(set(items["category"].astype(str))):
    rows=np.flatnonzero(pc==cat)
    if not len(rows): continue
    t=time.perf_counter()
    sub=items[items["category"].astype(str)==cat].reset_index(drop=True)
    X[rows]=build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), with_neighbours=True)
    log(f"  {cat}: {len(rows):,} пар, {len(sub):,} карточек за {time.perf_counter()-t:.0f}с")
    del sub; gc.collect()
np.save("/kaggle/working/features_llm2m.npy",X)
log("признаки 2 млн сохранены")

Xh=np.load(prev+"/features_human.npy")
E1=np.load(prev+"/features_eval_lex.npy"); E2=np.load(prev+"/features_eval_mixed.npy")
hm=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"])
ev1=pd.read_parquet(base+"/eval_pairs.parquet"); ev2=pd.read_parquet(base+"/eval_pairs_mixed.parquet")
yh=hm["target"].to_numpy(np.int8); yl=pairs["label"].to_numpy(np.int8)
tm,vm=product_disjoint_pair_masks(hm["id1"].to_numpy(),hm["id2"].to_numpy(),0,3)
rel=np.flatnonzero(~vm)
c1=ev1["category"].astype(str).to_numpy(); c2=ev2["category"].astype(str).to_numpy()
y1=ev1["target"].to_numpy(np.int8); y2=ev2["target"].to_numpy(np.int8)
def macro(p,c,y): return float(np.mean([average_precision_score(y[c==k],p[c==k])
    for k in np.unique(c) if len(np.unique(y[c==k]))>1]))
P=dict(max_iter=800,learning_rate=0.05,max_leaf_nodes=63,random_state=0,early_stopping=False)
for tag,n in (("400 тыс",400_000),("1 млн",1_000_000),("2 млн",len(X))):
    m=HistGradientBoostingClassifier(**P).fit(np.vstack([X[:n],Xh[rel]]),
                                             np.concatenate([yl[:n],yh[rel]]))
    p1=m.predict_proba(E1)[:,1]; p2=m.predict_proba(E2)[:,1]
    np.save(f"/kaggle/working/pred2m_{n}_lex.npy",p1); np.save(f"/kaggle/working/pred2m_{n}_mix.npy",p2)
    log(f"{tag:<10} лексич {macro(p1,c1,y1):.6f}  смеш {macro(p2,c2,y2):.6f}")
log("готово")
